# Aptitud para café y riesgo de roya y muerte descendente (Phoma) en el Cauca

Este notebook usa Google Earth Engine para generar tres mapas del departamento del Cauca:

1. **Zonas aptas para café** (análisis multicriterio de clima, suelo y relieve).
2. **Riesgo de roya** (*Hemileia vastatrix*).
3. **Riesgo de muerte descendente** (*Phoma* spp.).

Los umbrales y pesos están en la celda **Criterios**. Son valores de partida basados en publicaciones de Cenicafé y deben calibrarse con datos de campo.

## 1. Conexión a Earth Engine

In [ ]:
import ee, geemap

PROYECTO_GEE = 'TU_PROYECTO_GEE'   # <-- cambia por tu proyecto de Google Cloud

ee.Authenticate()
ee.Initialize(project=PROYECTO_GEE)

## 2. Criterios (edita aquí umbrales y pesos)

- `trapecio: [a, b, c, d]`: puntaje 0 por debajo de `a`, sube a 1 en `b`, se mantiene hasta `c` y baja a 0 en `d`.
- `sube: [a, b]`: puntaje 0 en `a` o menos, 1 en `b` o más.
- `baja: [a, b]`: puntaje 1 en `a` o menos, 0 en `b` o más.
- Los pesos de cada mapa deben sumar 1.

In [ ]:
PERIODO = ('2015-01-01', '2025-01-01')   # periodo para lluvia, humedad y viento

APTITUD_CAFE = {
    'temperatura': {'trapecio': [16, 18, 22, 24],         'peso': 0.30},  # °C, media anual
    'lluvia':      {'trapecio': [1200, 1500, 2800, 3500], 'peso': 0.20},  # mm/año
    'altitud':     {'trapecio': [1000, 1300, 1900, 2200], 'peso': 0.15},  # m
    'pendiente':   {'baja':     [50, 100],                'peso': 0.15},  # %
    'ph':          {'trapecio': [4.5, 5.0, 5.5, 6.5],     'peso': 0.20},  # pH del suelo
}

RIESGO_ROYA = {
    'temperatura': {'trapecio': [15, 21, 25, 28], 'peso': 0.40},  # °C, rango favorable al hongo
    'altitud':     {'baja':     [1600, 2000],     'peso': 0.25},  # m, más fuerte por debajo de 1.600
    'lluvia':      {'sube':     [1000, 2000],     'peso': 0.20},  # mm/año, mojado foliar
    'humedad':     {'sube':     [70, 85],         'peso': 0.15},  # % humedad relativa
}

RIESGO_PHOMA = {
    'altitud':     {'sube': [1400, 1800], 'peso': 0.30},  # m, limitante por encima de 1.600
    'temperatura': {'baja': [18, 21],     'peso': 0.20},  # °C, favorecida por debajo de 20
    'amplitud':    {'sube': [8, 11],      'peso': 0.20},  # °C entre día y noche
    'viento':      {'sube': [1.0, 1.7],   'peso': 0.15},  # m/s (6 km/h = 1,7 m/s)
    'lluvia':      {'sube': [1800, 3000], 'peso': 0.15},  # mm/año, lluvias prolongadas
}

# Clases de ESA WorldCover que se excluyen: construido, nieve, agua, humedal, manglar
COBERTURAS_EXCLUIDAS = [50, 70, 80, 90, 95]
EXCLUIR_AREAS_PROTEGIDAS = True

for nombre, crit in [('aptitud', APTITUD_CAFE), ('roya', RIESGO_ROYA), ('phoma', RIESGO_PHOMA)]:
    total = sum(v['peso'] for v in crit.values())
    assert abs(total - 1) < 1e-6, f'Los pesos de {nombre} suman {total}, deben sumar 1'

## 3. Área de estudio

In [ ]:
cauca = (ee.FeatureCollection('FAO/GAUL/2015/level1')
         .filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))
         .filter(ee.Filter.eq('ADM1_NAME', 'Cauca')))   # nombre exacto: excluye Valle del Cauca
region = cauca.geometry()
print('Departamentos encontrados:', cauca.size().getInfo())   # debe ser 1

## 4. Capas de clima, suelo y relieve

In [ ]:
import math

# Relieve (SRTM, 30 m)
dem = ee.Image('USGS/SRTMGL1_003')
altitud = dem.rename('altitud')
pendiente = (ee.Terrain.slope(dem).multiply(math.pi / 180).tan()
             .multiply(100).rename('pendiente'))            # en %

# Climatología WorldClim (1 km): temperatura media y amplitud térmica diaria
bio = ee.Image('WORLDCLIM/V1/BIO')
temperatura = bio.select('bio01').multiply(0.1).rename('temperatura')   # °C
amplitud = bio.select('bio02').multiply(0.1).rename('amplitud')         # °C

# Lluvia anual media (CHIRPS)
anios = ee.Number(ee.Date(PERIODO[1]).difference(ee.Date(PERIODO[0]), 'year'))
lluvia = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterDate(*PERIODO).sum().divide(anios).rename('lluvia'))   # mm/año

# Humedad relativa y viento (ERA5-Land mensual, ~11 km)
era = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
       .filterDate(*PERIODO).mean())
t = era.select('temperature_2m').subtract(273.15)
td = era.select('dewpoint_temperature_2m').subtract(273.15)
def presion_vapor(x):
    return x.multiply(17.625).divide(x.add(243.04)).exp()
humedad = presion_vapor(td).divide(presion_vapor(t)).multiply(100).rename('humedad')  # %
viento = (era.select('u_component_of_wind_10m').pow(2)
          .add(era.select('v_component_of_wind_10m').pow(2)).sqrt().rename('viento'))  # m/s

# pH del suelo en superficie (OpenLandMap, 250 m)
ph = (ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02')
      .select('b0').multiply(0.1).rename('ph'))

# Cobertura del suelo (ESA WorldCover, 10 m)
cobertura = ee.ImageCollection('ESA/WorldCover/v200').first().rename('cobertura')

capas = (ee.Image.cat([altitud, pendiente, temperatura, amplitud,
                       lluvia, humedad, viento, ph]).clip(region))
print(capas.bandNames().getInfo())

## 5. Funciones de puntaje

In [ ]:
def puntaje(img, regla):
    """Convierte una capa a un puntaje de 0 a 1 según la regla del criterio."""
    if 'trapecio' in regla:
        a, b, c, d = regla['trapecio']
        sube = img.subtract(a).divide(b - a)
        baja = ee.Image.constant(d).subtract(img).divide(d - c)
        return sube.min(baja).clamp(0, 1)
    if 'sube' in regla:
        a, b = regla['sube']
        return img.subtract(a).divide(b - a).clamp(0, 1)
    if 'baja' in regla:
        a, b = regla['baja']
        return ee.Image.constant(b).subtract(img).divide(b - a).clamp(0, 1)
    raise ValueError(f'Regla no reconocida: {regla}')

def indice(criterios, nombre):
    """Suma ponderada de los puntajes de cada variable (resultado de 0 a 1)."""
    total = ee.Image.constant(0)
    for variable, regla in criterios.items():
        total = total.add(puntaje(capas.select(variable), regla).multiply(regla['peso']))
    return total.rename(nombre).clip(region)

def clasificar(img):
    """Clases 1 a 5 con cortes en 0,2 / 0,4 / 0,6 / 0,8."""
    return (img.gte(0.2).add(img.gte(0.4)).add(img.gte(0.6)).add(img.gte(0.8))
            .add(1).rename('clase').updateMask(img.mask()))

## 6. Restricciones

In [ ]:
permitido = cobertura.remap(COBERTURAS_EXCLUIDAS, [0] * len(COBERTURAS_EXCLUIDAS), 1)

if EXCLUIR_AREAS_PROTEGIDAS:
    protegidas = (ee.FeatureCollection('WCMC/WDPA/current/polygons')
                  .filterBounds(region))
    mascara_protegidas = ee.Image.constant(1).paint(protegidas, 0)
    permitido = permitido.And(mascara_protegidas)

permitido = permitido.clip(region)

## 7. Mapa de zonas aptas para café

In [ ]:
aptitud = indice(APTITUD_CAFE, 'aptitud').updateMask(permitido)
aptitud_clase = clasificar(aptitud)

PALETA_APT = ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
ETIQ_APT = ['No apto', 'Marginal', 'Moderado', 'Apto', 'Muy apto']

m = geemap.Map()
m.centerObject(cauca, 8)
m.addLayer(aptitud_clase, {'min': 1, 'max': 5, 'palette': PALETA_APT}, 'Aptitud café')
m.addLayer(cauca.style(color='black', fillColor='00000000'), {}, 'Cauca')
m.add_legend(title='Aptitud para café', legend_dict=dict(zip(ETIQ_APT, PALETA_APT)))
m

## 8. Mapas de riesgo de roya y de Phoma

El riesgo se muestra solo donde la aptitud para café es al menos marginal (índice ≥ 0,2).

In [ ]:
zona_cafe = aptitud.gte(0.2)

roya = indice(RIESGO_ROYA, 'riesgo_roya').updateMask(zona_cafe)
phoma = indice(RIESGO_PHOMA, 'riesgo_phoma').updateMask(zona_cafe)
roya_clase = clasificar(roya)
phoma_clase = clasificar(phoma)

PALETA_RIESGO = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
ETIQ_RIESGO = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
vis = {'min': 1, 'max': 5, 'palette': PALETA_RIESGO}

m2 = geemap.Map()
m2.centerObject(cauca, 8)
m2.addLayer(roya_clase, vis, 'Riesgo de roya')
m2.addLayer(phoma_clase, vis, 'Riesgo de Phoma', False)   # actívala en el control de capas
m2.addLayer(cauca.style(color='black', fillColor='00000000'), {}, 'Cauca')
m2.add_legend(title='Riesgo', legend_dict=dict(zip(ETIQ_RIESGO, PALETA_RIESGO)))
m2

## 9. Área por clase (hectáreas)

In [ ]:
import pandas as pd

def area_por_clase(clase, etiquetas, escala=100):
    img = ee.Image.pixelArea().divide(10000).addBands(clase)
    res = img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='clase'),
        geometry=region, scale=escala, maxPixels=1e10, tileScale=4).getInfo()
    filas = [{'clase': etiquetas[int(g['clase']) - 1], 'hectareas': round(g['sum'])}
             for g in res['groups']]
    return pd.DataFrame(filas)

print('Aptitud para café');  display(area_por_clase(aptitud_clase, ETIQ_APT))
print('Riesgo de roya');     display(area_por_clase(roya_clase, ETIQ_RIESGO))
print('Riesgo de Phoma');    display(area_por_clase(phoma_clase, ETIQ_RIESGO))

## 10. Exportar a Google Drive (GeoTIFF)

In [ ]:
salida = ee.Image.cat([aptitud, roya, phoma]).toFloat()

tarea = ee.batch.Export.image.toDrive(
    image=salida, description='cauca_cafe_aptitud_riesgo',
    folder='cauca_gee', region=region, scale=100, crs='EPSG:4326', maxPixels=1e10)
tarea.start()
print('Exportación iniciada. Revisa el avance en https://code.earthengine.google.com/tasks')